In [1]:
%pip install matplotlib numpy torch

  Using cached matplotlib-3.10.8-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp313-cp313-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.8-cp313-cp313-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp313-cp313-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl (73 kB)
Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl (7.1 MB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ----- ---------------------------------- 1/7 [pillow]
   ----


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
from data.h36m_dataset import H36MDataset, build_dataloaders
from utils.metrics import mpjpe, mpjpe_at_horizons

In [2]:
# Build data loaders
# Update this path to match your local H3.6M dataset location
DATA_PATH = "D:/L4S2/Research Project in AI/Research/iccv21_git_src/data_3d_h36m.npz"

train_loader, test_loader = build_dataloaders(
    DATA_PATH,
    batch_size=32,
    t_obs=10,
    t_pred=25,
    train_stride=5,
    test_stride=1
)

c:\Users\gayuni\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


[H36MDataset] 38045 windows | subjects=['S1', 'S5', 'S6', 'S7', 'S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9', 'S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz


### Evaluate MPJPE

In [3]:
# Sanity check 1: zero error on perfect prediction
B, T, J = 32, 25, 17
perfect_pred = torch.randn(B, T, J, 3)
result = mpjpe(perfect_pred, perfect_pred)
assert result.max() < 1e-5, "MPJPE should be 0 for perfect prediction"
print("✓ Sanity check 1 passed: zero error on perfect prediction")

✓ Sanity check 1 passed: zero error on perfect prediction


In [4]:
# Sanity check 2: random prediction gives non-zero error
random_pred = torch.randn(B, T, J, 3)
random_target = torch.randn(B, T, J, 3)
result = mpjpe(random_pred, random_target)
assert result.mean() > 0, "MPJPE should be non-zero for random predictions"
print(f"✓ Sanity check 2 passed: random error = {result.mean():.1f}mm")

✓ Sanity check 2 passed: random error = 2254.6mm


In [5]:
# Sanity check 3: compute on actual data batch
batch = next(iter(train_loader))
obs = batch['observed']    # (32, 10, 17, 3)
fut = batch['future']      # (32, 25, 17, 3)

# Simulate a "zero-velocity" baseline:
# predict last observed frame repeated T_pred times
last_obs = obs[:, -1:, :, :]              # (32, 1, 17, 3)
zero_vel_pred = last_obs.repeat(1, 25, 1, 1)  # (32, 25, 17, 3)

zv_results = mpjpe_at_horizons(zero_vel_pred, fut)
print("\nZero-velocity baseline MPJPE (mm):")
for ms, err in zv_results.items():
    print(f"  {ms}ms: {err:.1f}mm")


Zero-velocity baseline MPJPE (mm):
  80ms: 24.6mm
  160ms: 47.6mm
  320ms: 87.6mm
  560ms: 136.3mm
  1000ms: 199.7mm


c:\Users\gayuni\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### Evaluate ADE + FDE

In [11]:
from utils.metrics import ade, fde, gravity_violation_rate

In [12]:
batch = next(iter(train_loader))

obs = batch['observed']
fut = batch['future']

# Zero velocity baseline
last_obs = obs[:, -1:, :, :]
pred = last_obs.repeat(1, 25, 1, 1)

print("ADE:", ade(pred, fut).item())
print("FDE:", fde(pred, fut).item())

ADE: 117.72502136230469
FDE: 211.01486206054688


Expected behavior
- ADE < FDE
- Values similar scale as MPJPE

### Evaluate GVR

In [13]:
# GVR on ground truth should be LOW (humans rarely violate gravity)
batch = next(iter(train_loader))
fut = batch['future']  # (32, 25, 17, 3)
gvr_gt = gravity_violation_rate(fut)
print(f"GVR on ground truth:      {gvr_gt:.4f} (should be low, <0.10)")

# GVR on random noise should be HIGH
random_poses = torch.randn_like(fut)
gvr_random = gravity_violation_rate(random_poses)
print(f"GVR on random noise:      {gvr_random:.4f} (should be high, >0.50)")

# GVR on zero-velocity baseline
last_obs = batch['observed'][:, -1:, :, :].repeat(1, 25, 1, 1)
gvr_zv = gravity_violation_rate(last_obs)
print(f"GVR on zero-vel baseline: {gvr_zv:.4f}")

GVR on ground truth:      0.5425 (should be low, <0.10)
GVR on random noise:      0.7350 (should be high, >0.50)
GVR on zero-vel baseline: 0.5312


Success criterion: GVR on ground truth is below 0.10. GVR on random noise is above 0.50. If ground truth GVR is high (GVR (GT) > 0.10), your ankle joint indices (lankle_idx 6, rankle_idx 3) are wrong - cross-check against JOINT_NAMES_17.